In [ ]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("langchain").setLevel(logging.ERROR)

#Requirements

In [ ]:
# Install dependencies
!pip install --upgrade langchain faiss-cpu sentence-transformers google-colab
!pip install langchain-core
!pip install pypdf
!pip install langchain-text-splitters
!pip install langchain_huggingface
!pip install langchain_community
!pip install langchain_groq

In [ ]:
GROQ_API_KEY="api_key"

#Document Loading:
We load the document either using simple text, or urls or using pdfs.

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader

# doc_text = """
# Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
# He was born on June 28, 1971, in Pretoria, South Africa.
# His major achievements include advancing space exploration and electric vehicles.
# Musk is also involved with Neuralink and The Boring Company.
# This document provides a brief overview of Musk's background and accomplishments.
# """

urls = [
    "https://arxiv.org/abs/1706.03762",
    "https://arxiv.org/abs/2005.14165"
]
loader = WebBaseLoader(urls)

document = loader.load()

#For pdf following method


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://arxiv.org/pdf/1706.03762.pdf")
documents = loader.load()
print("Document Loading done")

#Text Splitter:
We split the text from documents into chunks so that we can create embeddings and store them in a vector database

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_split = splitter.split_documents(documents)

#Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    show_progress=False  # ✅ disables the widget
)

In [ ]:
from langchain_community.vectorstores import FAISS

We use FAISS for creating vector store with similarity with the documents

In [ ]:
vectorstore = FAISS.from_documents(docs_split, embeddings)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_tokens=1024,
    api_key=GROQ_API_KEY,
)

In [ ]:
from langchain_classic.chains import RetrievalQA

#Retriever

In [ ]:
retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,  # to get source documents with answers
    chain_type="stuff",            # concatenates retrieved docs into prompt
)

#Generating Output

In [ ]:
query = "What is the attention mechanism in transformers?"

result = qa_chain.invoke({"query": query})

print("Answer:", result['result'])
# print("\nSource Documents:")
# for doc in result['source_documents']:
#     print(f"- {doc.page_content}")

In [ ]:
!git remote add origin https://github.com/AyaanK123/RAG_Langchain.git
!git branch -M main
!git push -u origin main